In [0]:
# import requests

# base_url = "https://d37ci6vzurychx.cloudfront.net/trip-data/yellow_tripdata_{}.parquet"
# months = ["2026-01", "2026-02","2026-03"] # , "2026-03", "2026-04"

# volume_path = "/Volumes/nycyellow/landing/raw/"

# for month in months:
#     try:
#         url = base_url.format(month)
#         # write straight into the Volume path — no /tmp staging needed
#         target_path = volume_path + f"yellow_tripdata_{month}.parquet"

#         response = requests.get(url, stream=True)
#         response.raise_for_status()

#         with open(target_path, "wb") as f:
#             for chunk in response.iter_content(chunk_size=8192):
#                 f.write(chunk)

#         print(f"Loaded {month}")
#     except requests.exceptions.HTTPError:
#         print(f"{month} not available yet — skipping")

In [0]:
import requests
from datetime import date

catalog = "nycyellow"
schema = "landing"
volume_path = f"/Volumes/{catalog}/{schema}/raw/"
base_url = "https://d37ci6vzurychx.cloudfront.net/trip-data/yellow_tripdata_{}.parquet"

def month_minus(d: date, months_back: int) -> str:
    """Returns a YYYY-MM string for 'months_back' months before date d."""
    month = d.month - months_back
    year = d.year
    while month <= 0:
        month += 12
        year -= 1
    return f"{year:04d}-{month:02d}"

# TLC publishes with a ~2 month lag, so today's "expected latest" month is today minus 2
target_month = month_minus(date.today(), 2)
target_filename = f"yellow_tripdata_{target_month}.parquet"
target_path = volume_path + target_filename
url = base_url.format(target_month)

# Idempotency check — skip if we already have this file
try:
    dbutils.fs.ls(target_path)
    print(f"{target_month} already ingested — skipping")
except Exception:
    # File doesn't exist yet in the Volume, so attempt download
    response = requests.get(url, stream=True)
    if response.status_code == 200:
        with open(target_path, "wb") as f:
            for chunk in response.iter_content(chunk_size=8192):
                f.write(chunk)
        print(f"Loaded {target_month}")
    elif response.status_code == 404:
        print(f"{target_month} not published yet — will retry on next scheduled run")
    else:
        response.raise_for_status()

In [0]:
# Separately, downloading the Taxi Zone lookup file (small, single file, no loop needed)
import requests


zone_url = "https://d37ci6vzurychx.cloudfront.net/misc/taxi_zone_lookup.csv"

catalog = "nycyellow"
schema = "landing"

file_path = f"/Volumes/{catalog}/{schema}/raw/zone_lookup/taxi_zone_lookup.csv"


response = requests.get(zone_url)
response.raise_for_status()

with open(file_path, "wb") as f:
    f.write(response.content)

print("Loaded taxi_zone_lookup.csv")